# Professor Tetraminx: воспроизводимый разбор решения

**Итог:** public score **28 456**, public rank **#1**, отрыв от второго места — **11 ходов**.  
Файл `outputs/submission_28456.csv` содержит 1 000 путей; каждый путь точно приводит официальное 88-позиционное состояние к identity.

Notebook воспроизводит проверку submission, score progression, построчные улучшения, PDB lower bounds и growth bound. Тяжёлые BFS/PDB/SAT-построения не перезапускаются: их проверенные результаты и исходники перечислены в конце.

## 1. Пути и зависимости

Notebook можно запускать из корня workspace или из каталога `outputs`. Нужны только Python, NumPy и pandas.

In [1]:
from pathlib import Path
from hashlib import sha256
from math import comb
import csv
import json

import numpy as np
import pandas as pd

def find_root() -> Path:
    starts = [Path.cwd(), Path.cwd().parent]
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / 'work/data/puzzle_info.json').is_file():
                return candidate
    raise FileNotFoundError('Не найден корень workspace с work/data/puzzle_info.json')

ROOT = find_root()
DATA = ROOT / 'work/data'
OUTPUTS = ROOT / 'outputs'
FINAL = OUTPUTS / 'submission_28456.csv'
ROOT

PosixPath('/Users/dmitrii/Documents/Codex/2026-08-04/https-www-kaggle-com-competitions-cayley')

## 2. Официальные генераторы

Состояние — pull-перестановка длины 88. Ход `g` применяется как `state = state[g]`. В данных 24 направленных хода: 12 осей порядка 3 и два направления для каждой оси.

In [2]:
info = json.loads((DATA / 'puzzle_info.json').read_text())
move_names = list(info['generators'])
name_to_index = {name: i for i, name in enumerate(move_names)}
moves = np.asarray([info['generators'][name] for name in move_names], dtype=np.uint8)
identity = np.arange(88, dtype=np.uint8)

with (DATA / 'test.csv').open(newline='') as handle:
    initial_states = {
        int(row['initial_state_id']): np.asarray(
            [int(x) for x in row['initial_state'].split(',')], dtype=np.uint8
        )
        for row in csv.DictReader(handle)
    }

assert moves.shape == (24, 88)
assert len(initial_states) == 1000
assert all(np.array_equal(np.sort(s), identity) for s in initial_states.values())

inverse_index = []
for name in move_names:
    inverse_name = name[1:] if name.startswith('-') else '-' + name
    inverse_index.append(name_to_index[inverse_name])
inverse_index = np.asarray(inverse_index, dtype=np.int16)

for i, g in enumerate(moves):
    assert np.array_equal(identity[g][moves[inverse_index[i]]], identity)
    assert np.array_equal(identity[g][g][g], identity)

print(f'Состояний: {len(initial_states)}, направленных ходов: {len(moves)}, осей: {len(moves)//2}')

Состояний: 1000, направленных ходов: 24, осей: 12


## 3. Точный verifier

Эвристика может только предложить путь. Принять его можно лишь после exact replay официальными перестановками.

In [3]:
def load_submission(path: Path) -> dict[int, tuple[int, ...]]:
    with path.open(newline='') as handle:
        rows = csv.DictReader(line for line in handle if line.strip())
        result = {}
        for row in rows:
            text = row['path'].strip()
            result[int(row['initial_state_id'])] = tuple(
                name_to_index[name] for name in text.split('.') if name
            )
    return result

def replay(initial: np.ndarray, word: tuple[int, ...]) -> np.ndarray:
    state = initial.copy()
    for move in word:
        state = state[moves[move]]
    return state

def validate(paths: dict[int, tuple[int, ...]]) -> dict[str, int]:
    if set(paths) != set(initial_states):
        missing = set(initial_states) - set(paths)
        extra = set(paths) - set(initial_states)
        raise ValueError(f'Неверные ID: missing={len(missing)}, extra={len(extra)}')
    invalid = [pid for pid, word in paths.items()
               if not np.array_equal(replay(initial_states[pid], word), identity)]
    if invalid:
        raise ValueError(f'Невалидные строки: {invalid[:10]}')
    return {'rows': len(paths), 'invalid': 0, 'score': sum(map(len, paths.values()))}

final_paths = load_submission(FINAL)
final_metrics = validate(final_paths)
final_metrics['sha256'] = sha256(FINAL.read_bytes()).hexdigest()
final_metrics

{'rows': 1000,
 'invalid': 0,
 'score': 28456,
 'sha256': '1269bb4aec8be2e7320031ceb796ead71c43c42980179f66785bd1c0d534f587'}

In [4]:
EXPECTED_SHA = '1269bb4aec8be2e7320031ceb796ead71c43c42980179f66785bd1c0d534f587'
assert final_metrics == {
    'rows': 1000,
    'invalid': 0,
    'score': 28456,
    'sha256': EXPECTED_SHA,
}
print('OK: 1000/1000 exact replay, score=28456, SHA-256 совпадает')

OK: 1000/1000 exact replay, score=28456, SHA-256 совпадает


## 4. Реальная последовательность улучшений

Каждый checkpoint ниже заново replay-проверяется. Это не вручную записанные score.

In [5]:
checkpoints = [
    ('Сильнейший полный public', ROOT / 'work/public_recovery/FINAL_tetraminx_28468.csv'),
    ('Построчный merge TPU', ROOT / 'work/candidates/submission_public_merged.csv'),
    ('Симметрия + инверсия', ROOT / 'work/candidates/submission_symmetry_revalidated.csv'),
    ('Exact radius 5', ROOT / 'work/candidates/submission_28458.csv'),
    ('Exact endgame: ID 994', ROOT / 'work/candidates/submission_28457.csv'),
    ('Exact endgame: ID 879', FINAL),
]

checkpoint_paths = []
rows = []
for stage, path in checkpoints:
    paths = load_submission(path)
    metrics = validate(paths)
    checkpoint_paths.append(paths)
    rows.append({'этап': stage, 'score': metrics['score'], 'валидно': metrics['rows']})
progression = pd.DataFrame(rows)
assert progression['score'].tolist() == [28468, 28463, 28459, 28458, 28457, 28456]
progression

,этап,score,валидно
0,Сильнейший полный public,28468,1000
1,Построчный merge TPU,28463,1000
2,Симметрия + инверсия,28459,1000
3,Exact radius 5,28458,1000
4,Exact endgame: ID 994,28457,1000
5,Exact endgame: ID 879,28456,1000


In [6]:
changes = []
for i in range(1, len(checkpoint_paths)):
    before, after = checkpoint_paths[i - 1], checkpoint_paths[i]
    for pid in sorted(before):
        if len(after[pid]) < len(before[pid]):
            changes.append({
                'переход': f'{sum(map(len, before.values()))} -> {sum(map(len, after.values()))}',
                'ID': pid,
                'было': len(before[pid]),
                'стало': len(after[pid]),
                'экономия': len(before[pid]) - len(after[pid]),
            })
changes_df = pd.DataFrame(changes)
assert changes_df['экономия'].sum() == 12
changes_df

,переход,ID,было,стало,экономия
0,28468 -> 28463,60,31,29,2
1,28468 -> 28463,991,29,28,1
2,28468 -> 28463,992,30,29,1
3,28468 -> 28463,993,29,28,1
4,28463 -> 28459,126,30,29,1
5,28463 -> 28459,256,29,28,1
6,28463 -> 28459,328,29,28,1
7,28463 -> 28459,749,29,28,1
8,28459 -> 28458,302,29,28,1
9,28458 -> 28457,994,29,28,1


### Что дало эти 12 ходов

- `28 468 -> 28 463`: post-row merge валидных TPU/public путей.
- `28 463 -> 28 459`: канонизация substrings под 24 тетраэдральными симметриями и group inversion.
- `28 459 -> 28 458`: полная таблица кратчайших слов до radius 5.
- `28 458 -> 28 456`: join с exact BFS endgame до depth 6.

Во всех случаях равенство замен проверялось как полная 88-позиционная перестановка.

## 5. Построчный merge

Score аддитивен по ID. Поэтому корректная стратегия — хранить самый короткий валидный путь для каждой строки независимо от источника.

In [7]:
def merge_shorter(base: dict[int, tuple[int, ...]], *candidates):
    best = dict(base)
    for candidate in candidates:
        validate(candidate)
        for pid, word in candidate.items():
            if len(word) < len(best[pid]):
                best[pid] = word
    validate(best)
    return best

merged = merge_shorter(checkpoint_paths[0], *checkpoint_paths[1:])
assert sum(map(len, merged.values())) == 28456
assert all(len(merged[pid]) == len(final_paths[pid]) for pid in merged)
print('Построчный merge всех checkpoint воспроизводит длины финального submission')

Построчный merge всех checkpoint воспроизводит длины финального submission


## 6. Exact endgame

Public `bfs_endgame.npz` — полный шар Кэли до distance 6. `local_endgame_opt.cpp` перечисляет короткий префикс от элемента окна и делает exact join с этой таблицей.

In [8]:
endgame_shells = pd.DataFrame({
    'depth': list(range(7)),
    'states': [1, 24, 408, 6592, 105136, 1659416, 26008172],
})
assert int(endgame_shells['states'].sum()) == 27_779_749
endgame_shells

,depth,states
0,0,1
1,1,24
2,2,408
3,3,6592
4,4,105136
5,5,1659416
6,6,26008172


## 7. Структура группы и admissible PDB

Точная декомпозиция:

```text
G = A12^3 × (2^5 : A6) × A4 × (C3)^4
|G| = 153829507379767660542689280000000
```

Ходы layer 2 действуют на A/B/D/E, layer 3 — на B/C/D, layer 4 — на A/T. 0/1-cost и weighted PDB дают ограничения на минимальные количества ходов каждого слоя; маленькая целочисленная оптимизация объединяет их без потери admissibility.

In [9]:
bound_files = [
    ('full-factor quotient', ROOT / 'work/current_pdb_bounds.csv'),
    ('первое coupling', ROOT / 'work/coupled_pdb_bounds.csv'),
    ('coupling с C-block', ROOT / 'work/coupled_cblock_pdb_bounds.csv'),
]
bound_rows = []
for label, path in bound_files:
    frame = pd.read_csv(path)
    bound_rows.append({
        'эвристика': label,
        'строк': len(frame),
        'сумма lower bound': int(frame['current_h'].sum()),
        'max per row': int(frame['current_h'].max()),
    })
bounds = pd.DataFrame(bound_rows)
assert bounds['сумма lower bound'].tolist() == [13884, 13885, 13893]
bounds

,эвристика,строк,сумма lower bound,max per row
0,full-factor quotient,1000,13884,17
1,первое coupling,1000,13885,17
2,coupling с C-block,1000,13893,17


## 8. Проверенные направления без дополнительной экономии


In [10]:
negative_results = pd.DataFrame([
    ('Exact radius-10 sweep', 'все 1000 строк', '0 ходов'),
    ('PDB bounded DFS', '600 окон + 2031 сертификат', '0 ходов'),
    ('Weighted PDB DFS', '60 окон исчерпаны', '0 ходов'),
    ('SAT/CEGAR', 'локальные окна и whole-row probe', '0 ходов'),
    ('Trajectory crossover', 'union + corridor radius 1/2/3', '0 ходов'),
    ('Plateau escape', 'равнодлинные геодезические замены', '0 ходов'),
    ('Neural beam/A*', 'ResMLP, Transformer, blend, frames, cuts', '0 ходов'),
    ('Stochastic beam', 'seeded noise + exact endgame', '0 ходов'),
    ('Bidirectional beam 16k/65k', '2869 exact + 40 bridge checks', '0 hits'),
    ('Multi-source substring mining', '5 независимых сильных submission', '0 ходов'),
    ('twips exact', '9 окон 12-><=11 + full probes', '0 ходов'),
    ('Новые public paths', '35/35 валидны', '0 строгих улучшений'),
], columns=['метод', 'проверенный объём', 'результат'])
negative_results

,метод,проверенный объём,результат
0,Exact radius-10 sweep,все 1000 строк,0 ходов
1,PDB bounded DFS,600 окон + 2031 сертификат,0 ходов
2,Weighted PDB DFS,60 окон исчерпаны,0 ходов
3,SAT/CEGAR,локальные окна и whole-row probe,0 ходов
4,Trajectory crossover,union + corridor radius 1/2/3,0 ходов
5,Plateau escape,равнодлинные геодезические замены,0 ходов
6,Neural beam/A*,"ResMLP, Transformer, blend, frames, cuts",0 ходов
7,Stochastic beam,seeded noise + exact endgame,0 ходов
8,Bidirectional beam 16k/65k,2869 exact + 40 bridge checks,0 hits
9,Multi-source substring mining,5 независимых сильных submission,0 ходов


## 9. Почему target <25 000 геометрически маловероятен

Для 12 генераторов порядка 3 граф коммутации имеет clique counts `1, 12, 30, 32, 17, 4`. Группа, в которой оставлены только отношения порядка 3 и известные коммутации, отображается на puzzle group. Поэтому её шары дают верхнюю границу для шаров puzzle. Growth series:

```text
W(z) = (1+2z)^5 / (1-14z-32z^2-32z^3)
```

In [11]:
GROUP_ORDER = 153829507379767660542689280000000
numerator = [comb(5, k) * 2**k for k in range(6)]
sphere = []
for n in range(27):
    value = numerator[n] if n < len(numerator) else 0
    if n >= 1:
        value += 14 * sphere[n - 1]
    if n >= 2:
        value += 32 * sphere[n - 2]
    if n >= 3:
        value += 32 * sphere[n - 3]
    sphere.append(value)

balls = np.cumsum(np.asarray(sphere, dtype=object)).tolist()
uniform_mean_lb = sum(1 - balls[d] / GROUP_ORDER for d in range(27))
growth = pd.DataFrame([
    {'radius': d, 'upper_bound_ball': balls[d], 'fraction_of_group': balls[d] / GROUP_ORDER}
    for d in [24, 25, 26]
])
assert abs(uniform_mean_lb - 26.717751703678083) < 1e-12
display(growth)
print(f'Строгая lower bound среднего расстояния uniform state: {uniform_mean_lb:.15f}')

,radius,upper_bound_ball,fraction_of_group
0,24,156915260064682784099072039361,0.001020
1,25,2527855598908305735264746691009,0.016433
2,26,40722960445644321792624560789953,0.264728


Строгая lower bound среднего расстояния uniform state: 26.717751703678083


Из bound следует `B25/|G| <= 1.643284%` и `E[D] >= 26.71775`. Это строгий результат для равномерного элемента, но **не** formal impossibility proof для конкретных 1 000 test states. Для fixed test нужны exact per-state distances или доказанная mixing bound. Он объясняет, почему average <25 не согласуется с геометрией хорошо перемешанных состояний.

## 10. Зафиксированный leaderboard snapshot

Последняя проверка выполнена через Kaggle API/CLI, без браузера, 2026-08-04 17:55:29 UTC:

| Rank | Участник | Score |
|---:|---|---:|
| 1 | Chekhlov Dmitrii | **28 456** |
| 2 | Andrew Lukyanenko | 28 467 |
| 3 | Tomas Rokicki | 28 481 |

Ни один no-gain кандидат не отправлялся. Лучший submission сохранён.

## 11. Исходники тяжёлых этапов

| Этап | Файл |
|---|---|
| Exact validator/merge | `work/validate_merge.py` |
| Symmetry + inversion | `work/symmetry_substring_optimizer.py` |
| Radius-5 table | `work/radius5_optimizer.py` |
| Radius-6/10 endgame | `work/local_endgame_opt.cpp` |
| PDB/IDA* | `work/pdb_bounded_dfs.cpp` |
| Full quotients | `work/full_factor_quotient.hpp` |
| SAT/CEGAR | `work/window_sat_optimize.py` |
| Corridor crossover | `work/corridor_crossover.py` |
| Neural beam | `work/mps_q_beam.py` |
| Neural A* | `work/neural_astar.py` |
| Bidirectional beam | `work/bidirectional_neural_beam.py` |
| twips translator | `work/twips_tetraminx.py` |

Подробная русская история и точные измерения: `outputs/SOLUTION_WRITEUP_RU.md`.

In [12]:
summary = {
    'public_score': final_metrics['score'],
    'public_rank': 1,
    'margin_to_second': 28467 - final_metrics['score'],
    'valid_rows': final_metrics['rows'],
    'sha256': final_metrics['sha256'],
}
summary

{'public_score': 28456,
 'public_rank': 1,
 'margin_to_second': 11,
 'valid_rows': 1000,
 'sha256': '1269bb4aec8be2e7320031ceb796ead71c43c42980179f66785bd1c0d534f587'}